In [17]:
import pandas as pd
import os
import numpy as np
import astropy
import astropy.units as u
from astropy.constants import M_jup, M_earth
import astropy.units as u
import matplotlib.pyplot as plt

def gen_orbits_csv(k, output_dir, output_file):
    k_proposal = int(k)
    os.makedirs(output_dir, exist_ok=True)

    rng = np.random.default_rng(42)
    T_BASELINE = 24.75
    M_mars_Mjup = (0.107 * M_earth).to(u.M_jup).value
    # Draw stellar masses (FGK), uniform
    mstar = rng.uniform(0.5, 1.5, size=k_proposal)
    # SMA_CRIT for a given star
    sma_crit = (T_BASELINE**2 * mstar)**(1/3)
    # Log-uniform draws
    sma_prop = 10**rng.uniform(np.log10(1), np.log10(100), size=k_proposal)
    mass_prop = 10**rng.uniform(np.log10(M_mars_Mjup), np.log10(80), size=k_proposal)

    # SNR proxy from William's eqn
    snr_proxy = (mass_prop / mstar) * sma_prop / (1.0 + (sma_prop / sma_crit)**3)
    accept_prob = snr_proxy / snr_proxy.max()
    accept = rng.uniform(size=k_proposal) < accept_prob


    sma = sma_prop[accept]
    mass = mass_prop[accept]
    mstar_accepted = mstar[accept]
    n = accept.sum()
    print(f"Accepted {n} / {k_proposal} = {n/k_proposal:.1%}")
    #ecc = rng.beta(a=0.867, b=3.03, size=n) - kipping
    ecc = rng.uniform(0, 0.99, size=n)
    inc = np.degrees(np.arccos(rng.uniform(0, 1, size=n)))
    Omega = rng.uniform(0, 360, size=n)
    omega = rng.uniform(0, 360, size=n)
    M_anom = rng.uniform(0, 360, size=n)

    labeled_data = {
        'sma': sma,
        'ecc': ecc,
        'mass_pl': mass,
        'mass_st': mstar_accepted,
        'inc': inc,
        'Omega': Omega,
        'omega': omega,
        'M_anom': M_anom,
    }


    csv_data = pd.DataFrame(labeled_data)
    output_fpath = os.path.join(output_dir, output_file)
    print(f"Writing output to {output_fpath}...")
    csv_data.to_csv(output_fpath, index=False)
    return csv_data


In [19]:
gen_orbits_csv(int(1e6), output_dir = '/Users/clarissardoo/Projects/Gaia_Planetsoutputs', output_file = 'orbits.csv')

Accepted 25133 / 1000000 = 2.5%
Writing output to /Users/clarissardoo/Projects/Gaia_Planets/outputs/orbits.csv...


,sma,ecc,mass_pl,mass_st,inc,Omega,omega,M_anom
0,8.376271,0.557441,58.403635,0.938878,52.815318,214.519597,268.355783,51.135353
1,5.906371,0.210554,8.836986,0.982303,10.206400,139.443659,175.070611,123.612334
2,3.655084,0.962593,0.589423,1.006330,89.049963,294.372248,140.670926,150.351484
3,1.398522,0.717215,41.703127,0.605897,40.187524,314.741301,246.604102,19.666786
4,1.460224,0.216030,28.969680,0.925882,40.975253,48.294736,234.402895,179.948136
...,...,...,...,...,...,...,...,...
25128,8.166208,0.059457,56.251249,1.123746,22.656347,208.847603,25.046948,46.011369
25129,4.808755,0.400262,17.046188,1.292331,87.048910,293.195916,211.675528,327.218264
25130,9.321783,0.173175,61.529110,0.884589,87.476733,152.446124,323.169017,257.888735
25131,3.686793,0.071719,46.433311,1.274216,45.101129,84.170260,58.790785,292.778577
